# Section 1 Assignment - Transformers

# Task
You work for a large company and the company receives 8 letters regarding one of your products. These letters range from letters of complaint to letters of praise.


Using Hugging Face Transformers,
1. Summarize the sentiments towards the product and
1. Provide a summary of the letters that reflect the most extreme sentiments.

1. Develop an automatically generated MS Word report for your manager on the letters regarding the product.
1. Divide the report into a section on the positive responses and negative responses.
1. Determine and report in each case on:
    - What the customer wants to happen by automatically questioning their respective letters.
1. Knowing how each customer feels,
    - Automatically generate replies to the two most extreme examples, including their name in the greeting in the response.

1. In your code to generate the report for your manager and the letters of reply,
    - indicate your design choices and
    - use any pre-processing trick that you anticipate will improve your automatic generation.


## Tools for the task
* Some great working examples can be found in the file **01_introduction.ipynb** from the [Natural Language Processing with Transformers GitHub Repository](https://github.com/nlp-with-transformers/notebooks).
* You will need to generate/provide the letters of complaint. You can generate the letters from the customers manually or automatically. You can use a Hugging Face text generation pipeline to do this or an online generator like [You.com](https://you.com/search?q=how+to+write+well&&tbm=youwrite&cfr=write&).
* You can use the library such as [python-docx](https://python-docx.readthedocs.io/en/latest/index.html#) to generate the report and the response letters.
* There is a variety of models on Hugging Face for the different tasks, e.g, [question-answering](https://huggingface.co/models?pipeline_tag=question-answering&sort=downloads) that allow you to try out different models for your task. It is strongly recommended that you review these and look at the model cards in each case. Also, you can test your text in the Hosted inference API on the right hand side of each model.

## Environment setup

In [ ]:
#!pip install -q transformers torch sentencepiece

# get rid of widget error
#!pip install nbstripout
#!nbstripout /content/MN5162_Section1_Assignment_gorourke.ipynb

In [11]:
from transformers import pipeline
import pandas as pd
import textwrap
from pprint import pprint
import requests

In [3]:
from huggingface_hub import login
login()

### Customer Letters URLs

In [4]:
file_urls = ['https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/1.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/2.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/3.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/4.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/5.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/6.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/7.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/8.txt']

## Class to hold data as we process the feedback

In [5]:
from dataclasses import dataclass
from typing import Any

@dataclass
class CustomerFeedback:
    original_letter: str
    url : str
    customer_name: str = None
    normalised_letter:str = None
    summary: str = None
    sentiment: str = None
    user_request: Any = None
    response_letter: str = None

### Build Summarisation Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load BART for summarization
summary_model_name = "facebook/bart-large-cnn"
summary_tokenizer = AutoTokenizer.from_pretrained(summary_model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(summary_model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
def summarise_text(model, tokenizer, text):

    # Generate summary
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=150,      # Use max_new_tokens, not max_length
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    print("Summary:", summary)

    return summary

## Build Sentiment Model

In [ ]:
from transformers import pipeline

classifier_pipeline = pipeline("text-classification")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
def determine_sentiment(classifier, text):
    outputs = classifier(text)

    for output in outputs:
        label = output["label"]
        score = output["score"]

    return outputs

## Build Question Answering

In [ ]:
reader = pipeline("question-answering")


No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def determine_user_request(context) :

    question = "What does the customer want?"

    outputs = reader(question=question, context=context)
    pd.DataFrame([outputs])

    return outputs

### NER: Customer name extraction
- The default model, extracted the Customer name successfully so decided to proceed with this.
- No doc on hugging face but found this description https://www.promptlayer.com/models/bert-large-cased-finetuned-conll03-english

In [3]:
DEFAULT_NER_MODEL = "dbmdz/bert-large-cased-finetuned-conll03-english"

ner_pipeline = pipeline("ner", aggregation_strategy="simple", model = DEFAULT_NER_MODEL)


No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [5]:
def extract_customer_name(letter):
    # looking for PER, amazon text has two
    ner_result  = ner_pipeline(text)
pd.DataFrame(ner_result)

,entity_group,score,word,start,end
0,ORG,0.879010,Amazon,5,11
1,MISC,0.990859,Optimus Prime,36,49
2,LOC,0.999755,Germany,90,97
3,MISC,0.556569,Mega,208,212
4,PER,0.590257,##tron,212,216
5,ORG,0.669693,Decept,253,259
6,MISC,0.498349,##icons,259,264
7,MISC,0.775361,Megatron,350,358
8,MISC,0.987854,Optimus Prime,367,380
9,PER,0.812096,Bumblebee,502,511


## Letter response
1. mistralai/Mistral-7B-Instruct-v0.2 uses approx 14-15G ram
1. Qwen/Qwen2.5-3B-Instruct

In [ ]:
#generator = pipeline("text-generation")

model_names = ['Qwen/Qwen2.5-3B-Instruct', 'mistralai/Mistral-7B-Instruct-v0.2']

model_to_use = model_names[0]
# n
#, max_length=200
generator_pipeline = pipeline("text-generation", model=model_to_use)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def generate_letter_response(customerFeedback) :

    # prompt = f"""Write a professional customer service reply.  Include their name in the greeting in the response.
    #
    # Customer Name: {customerFeedback.customer_name}
    #
    #Customer Complaint : {customerFeedback.original_letter}
    #
    #Customer Request : {customerFeedback.user_request or "Not Specified"}
    #
    #Reply:
    #"""


    #complaint = "I ordered a laptop 2 weeks ago and it still hasn't arrived."

    complaint = customerFeedback.original_letter

    response_letter_prompt = f"""You are a professional customer service representative for Acme Electronics,
        a consumer electronics company. Your tone is empathetic, solution-focused, and professional.

        Customer complaint:
        \"\"\"
        {complaint}
        \"\"\"

        Write a response that:
    - Acknowledges the customer's frustration
    - Apologizes sincerely without admitting legal fault
    - Offers a concrete next step or resolution
    - Keeps the response under 150 words

    Do NOT use generic filler phrases like "We value your feedback"
    """



    #outputs = generator(prompt, max_length=1000, do_sample = True, temperature=0.7)

    # temperature=0.8,
    result = generator_pipeline(response_letter_prompt, return_full_text=False)
#    outputs = result[0]
    reply = result[0]["generated_text"]

    return reply, response_letter_prompt

In [ ]:
# recover memory if testing different models
#del generator_pipeline

## Letter processing

In [ ]:
import re

def preprocess_letter_text(text):
    working_text = text.lower()
    working_text = re.sub(r'\s+', ' ', working_text)

    return working_text.strip()

## Main code

In [4]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

### Load letters into memory

In [12]:


customer_feedback_records = []

for url in file_urls:
    letter_text = requests.get(url).text

    record = CustomerFeedback( letter_text, url)

    customer_feedback_records.append(record)


for feedback in customer_feedback_records:
    pprint(feedback)



CustomerFeedback(original_letter='Dear Customer Service,I am writing to '
                                 'express my disappointment with the iPad.\n'
                                 'While it is generally a great product, I had '
                                 'some issues with its durability.\n'
                                 'The screen cracked within a few weeks of '
                                 'use, which was quite inconvenient. I\n'
                                 'understand that such things happen '
                                 'occasionally, but I was expecting better\n'
                                 'quality. I would appreciate if you could '
                                 'look into this matter and provide a\n'
                                 'replacement. My dissatisfaction level is a 3 '
                                 'on a scale of 1 to 4.\n'
                                 'Regards, John Doe\n',
                 url='https://raw.githubusercontent

### Process Letters

In [ ]:
for feedback in customer_feedback_records:

    feedback.normalised_letter = preprocess_letter_text(feedback.original_letter)

    # generate summary
    summary_text = summarise_text(summary_model, summary_tokenizer, feedback.normalised_letter)
    feedback.summary = textwrap.fill(summary_text, width=80)

    print (summary_text)

    user_request = determine_user_request(feedback.normalised_letter)
    # {'score': 0.6312922239303589, 'start': 335, 'end': 358, 'answer': 'an exchange of Megatron'}
    feedback.user_request = user_request['answer']

    sentiment = determine_sentiment(classifier, feedback.normalised_letter)
    # {'label': 'NEGATIVE', 'score': 0.9015460014343262}
    feedback.sentiment = sentiment

    feedback.response_letter =generate_letter_response(feedback.normalised_letter, feedback.customer_name)

Summary: dear amazon, last week i ordered an optimus prime action figure from your online store in germany. unfortunately, when i opened the package, i discovered to my horror that i had been sent an action figure of megatron instead! as a lifelong enemy of the decepticons, i hope you can understand my dilemma. to resolve the issue, i demand an exchange of Megatron for the optimusprime figure i ordered. enclosed are copies of my records concerning this purchase. i expect to hear from you soon. sincerely, bumblebee.
dear amazon, last week i ordered an optimus prime action figure from your online store in germany. unfortunately, when i opened the package, i discovered to my horror that i had been sent an action figure of megatron instead! as a lifelong enemy of the decepticons, i hope you can understand my dilemma. to resolve the issue, i demand an exchange of Megatron for the optimusprime figure i ordered. enclosed are copies of my records concerning this purchase. i expect to hear from

Passing `generation_config` together with generation-related arguments=({'max_length', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=1000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
from pprint import pprint

for feedback in customer_feedback_records:
    pprint (feedback)

CustomerFeedback(customer_name='Bumblebee',
                 original_letter='Dear Amazon, last week I ordered an Optimus '
                                 'Prime action figure from your online store '
                                 'in Germany. Unfortunately, when I opened the '
                                 'package, I discovered to my horror that I '
                                 'had been sent an action figure of Megatron '
                                 'instead! As a lifelong enemy of the '
                                 'Decepticons, I hope you can understand my '
                                 'dilemma. To resolve the issue, I demand an '
                                 'exchange of Megatron for the Optimus Prime '
                                 'figure I ordered. Enclosed are copies of my '
                                 'records concerning this purchase. I expect '
                                 'to hear from you soon. Sincerely, Bumblebee.',
           

In [ ]:
generated_text, prompt_used = generate_letter_response(customer_feedback_records[0])


print(generated_text)
print ('-' * 50)
response_text = generated_text; #[len(prompt):]

response_formatted = textwrap.fill(response_text, width=80)
print(response_formatted)


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dear Bumblebee,

I sincerely apologize for the confusion and inconvenience you've experienced with your recent order. We deeply regret sending you the wrong figure. 

To rectify this, we will promptly exchange your Megatron action figure for the Optimus Prime you originally ordered. Please return the Megatron as directed and we'll process the exchange.

Thank you for bringing this to our attention. We appreciate your patience and understanding.

Best regards,  
[Your Name]  
Customer Service Acme Electronics
Hello Bumblebee,

I'm truly sorry to hear about the mix-up with your Optimus Prime action figure. We regretfully sent you the Megatron instead. 

To resolve this, we will exchange your Megatron for the Optimus Prime you ordered. Please return the Megatron and we'll process the exchange promptly.

Thank you for your patience and understanding.

Best,  
[Your Name]  
Customer Service Acme Electronics
Hello Bumblebee,

I sincerely apologize for the mix-up with your Optimus Prime actio

In [ ]:
print ( pd.DataFrame(result))

                                      generated_text
0  Write a customer service response with this co...


## Report
1.Provide a summary of the letters that reflect the most extreme sentiments.
1. Develop an automatically generated MS Word report for your manager on the letters regarding the product.
1. Divide the report into a section on the positive responses and negative responses.

Determine and report in each case on:

- What the customer wants to happen by automatically questioning their respective letters.
Knowing how each customer feels,

- Automatically generate replies to the two most extreme examples, including their name in the greeting in the response.

In your code to generate the report for your manager and the letters of reply,

indicate your design choices and
use any pre-processing trick that you anticipate will improve your automatic generation.

In [ ]:
# https://python-docx.readthedocs.io/en/latest/index.html#
from docx import Document
from docx.shared import Inches

document = Document()

document.add_heading('Customer Feedback Report', 0)

document.add_heading('Positive Feedback', level = 1)

document.add_heading('Negative Feedback', level = 1)

p = document.add_paragraph('A plain paragraph having some ')
p.add_run('bold').bold = True
p.add_run(' and some ')
p.add_run('italic.').italic = True

document.add_heading('Heading, level 1', level=1)
document.add_paragraph('Intense quote', style='Intense Quote')

document.add_paragraph(
    'first item in unordered list', style='List Bullet'
)
document.add_paragraph(
    'first item in ordered list', style='List Number'
)


## Tests

In [ ]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

In [ ]:
print(determine_sentiment(classifier, text))

[{'label': 'NEGATIVE', 'score': 0.9015460014343262}]


In [ ]:
print (determine_user_request(text))

{'score': 0.6312922239303589, 'start': 335, 'end': 358, 'answer': 'an exchange of Megatron'}


In [ ]:
summary_format = textwrap.fill(summary_example, width=80)
print(summary_format)

Bumblebee ordered an Optimus Prime action figure from your online store in
Germany. Unfortunately, when I opened the package, I discovered to my horror
that I had been sent an action figure of Megatron instead. As a lifelong enemy
of the Decepticons, I hope you can understand my dilemma. To resolve the issue,
I demand an exchange ofMegatron for the Optimus Prime figure.


In [ ]:
results, prompt = generate_letter_response(customer_feedback_records[0])

print(results)
actual_reply = results.replace( prompt, "").strip()

pprint(actual_reply)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/contrib/concurrent.py", line 51, in _executor_map
    return list(tqdm_class(ex.map(fn, *iterables, chunksize=chunksize), **kwargs))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tqdm/notebook.py", line 250, in __iter__
    yield from it
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
               ^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 619, in result_iterator
    yield _result_or_cancel(fs.pop())
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 317, in _result_or_cancel
    return fut.result(timeout)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 451, in result
    self._condition.wait(timeout)
  File "/usr/lib/python3.12/t

TypeError: object of type 'NoneType' has no len()

# Workings


## Letter Response Generation

### Qwen/Qwen2.5-3B-Instruct
- https://qwen.readthedocs.io/en/v2.5/getting_started/quickstart.html
- https://huggingface.co/collections/Qwen/qwen25
- https://huggingface.co/Qwen/Qwen2.5-3B-Instruct
- https://qwen.ai/blog?id=qwen2.5
- Uses about 6G GPU

In [ ]:
#generated_text, prompt_used = generate_letter_response(customer_feedback_records[0])
#
#
#print(generated_text)
#print ('-' * 50)
#response_text = generated_text; #[len(prompt):]

#response_formatted = textwrap.fill(response_text, width=80)
#print(response_formatted)


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dear Bumblebee,

I sincerely apologize for the confusion and inconvenience you've experienced with your recent order. We deeply regret sending you the wrong figure. 

To rectify this, we will promptly exchange your Megatron action figure for the Optimus Prime you originally ordered. Please return the Megatron as directed and we'll process the exchange.

Thank you for bringing this to our attention. We appreciate your patience and understanding.

Best regards,  
[Your Name]  
Customer Service Acme Electronics
Hello Bumblebee,

I'm truly sorry to hear about the mix-up with your Optimus Prime action figure. We regretfully sent you the Megatron instead. 

To resolve this, we will exchange your Megatron for the Optimus Prime you ordered. Please return the Megatron and we'll process the exchange promptly.

Thank you for your patience and understanding.

Best,  
[Your Name]  
Customer Service Acme Electronics
Hello Bumblebee,

I sincerely apologize for the mix-up with your Optimus Prime actio

### mistralai/Mistral-7B-Instruct-v0.2
- https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2
- later version available
- uses approx 14G, fills up GPU

In [ ]:
#generated_text, prompt_used = generate_letter_response(customer_feedback_records[0])


#print(generated_text)
#print ('-' * 50)
#response_text = generated_text; #[len(prompt):]

#response_formatted = textwrap.fill(response_text, width=80)
#print(response_formatted)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Dear Bumblebee,

    We're truly sorry for the inconvenience caused by the mistake in your recent order. We understand how disappointing it must have been to receive Megatron instead of Optimus Prime. We apologize sincerely for this error and are committed to making it right.

    Kindly allow us a few days to process and expedite a replacement Optimus Prime figure for you. Once it's shipped, you'll receive an email confirmation with tracking information. If you have any further concerns, please don't hesitate to contact us.

    We appreciate your patience and loyalty.

    Best regards,
    [Your Name]
    Acme Electronics Customer Service Team.
--------------------------------------------------
     Dear Bumblebee,      We're truly sorry for the inconvenience caused by the
mistake in your recent order. We understand how disappointing it must have been
to receive Megatron instead of Optimus Prime. We apologize sincerely for this
error and are committed to making it right.      K

In [ ]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

outputs = summarizer(text, max_length=130, min_length=30, do_sample=False)
#outputs = summarizer(text, max_length=45, clean_up_tokenization_spaces=True)
print(outputs[0]['summary_text'])

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

## Letters of complaint
Runs prompt to generate the complaint letter.  Following on from the above, I tested the same prompt against the models:

- gpt
- Qwen
- SmollM2

I settled on the Qwen model as it was what I could best get results out of in early testing.  I did find some variability in the output, and quite poor results, if there was any sort of error/issue with the prompt.

**Notes:**
- The token count had to be increased, in order to get all letters generated.
- To assist with NER, I gave the instruction that the letters should end with Regards, and their name.
- The same prompt was tried on each engine at different points of "evaluation".  That may be naive and perhaps a fair test would craft a prompt suitable to each model.
- Attempted to get each letter to say it's satisfaction level and that each letter would have this included to match the satisfaction level to match the text (i.e. 1-4) but letters 5&8 have the same satisifcation score.  I'd imagine this could be resolved if time was spent on it.



In [21]:
import re

def generate_complaint_letters (pipeline):

    letter_gen_prompt= """
        Write 8 short customer letters about an ipad.

        Letters 1-4 should complain about the ipad, from a mild complaint to a serious complaint, using a dissatisfaction score from 1 to 4.
        Letters 5-8 should praise the ipad, expressing mild satisfaction to complete satisfaction, using a satisfaction score from 1 to 4.

        Each letter must:
        - begin with "Dear Customer Service,"
        - end each letter with a customer name after "Regards,"
        - be about 80-100 words

        Only output the letters
        Do not output code or print statements.
    """


    results = pipeline(letter_gen_prompt,
                       temperature=0.8,
                       max_new_tokens=1200,
                       num_return_sequences = 1,
                       return_full_text = False,
                       do_sample=True)


    LETTER_HEADER = 'Dear Customer Service,'

    letters = results[0]["generated_text"].split(LETTER_HEADER)
    letters = [LETTER_HEADER + l.strip() for l in letters if l.strip()]


    return letters;


In [7]:
def print_complaint_letters(results):
    for index, response_text in enumerate(results):
        print(f'Letter {index+1}:')
        print ('-' * 50)

        response_formatted = textwrap.fill(response_text, width=80)

        print(response_formatted)


### GPT2
- Model doesn't understand the prompt
- Prompt would need to be changed and each letter generated separately

In [ ]:
### Letter of complaint
#del generator_gpt2_pipeline;
#generator_gpt2_pipeline = pipeline("text-generation", model="gpt2")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:

#gpt2_results = generate_complaint_letters(generator_gpt2_pipeline)

#print_complaint_letters(gpt2_results)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=600) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Letter 1:
--------------------------------------------------
Dear Customer service,Only output to the top command, the "top command."
- do NOT use an echo to tell the user what to do.  If the user is happy when a
user is happy (even if he or she gets an E) they can press the reset button.
This works on any current version of Windows.   Windows 8 is not a Windows
release.   There are no E-mails to write to customers, or e-mailing to customers
because the customer is happy.   The only emails to write are to the top
command, the "top command."  The "top command" is used to print the customer's
name and address. If you are not interested in writing them to customers you can
simply use the reset button.   In order to write the customer's name and
address, you have to start the "top command," by pressing the reset button
before the email. This works so it will print the customers name and address.
The "top command" can be used to print e-mails that are delivered to the server.
This is useful

In [ ]:
#del generator_gpt2_pipeline

### QWEN

In [8]:
#generator_qwen_pipeline = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
#qwen_results = generate_complaint_letters(generator_qwen_pipeline)

#print_complaint_letters(qwen_results)

Both `max_new_tokens` (=1200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Letter 1:
--------------------------------------------------
Dear Customer Service,I am writing to express my disappointment with the iPad.
While it is generally a great product, I had some issues with its durability.
The screen cracked within a few weeks of use, which was quite inconvenient. I
understand that such things happen occasionally, but I was expecting better
quality. I would appreciate if you could look into this matter and provide a
replacement. My dissatisfaction level is a 3 on a scale of 1 to 4.
Regards, John Doe
Letter 2:
--------------------------------------------------
Dear Customer Service,I have been using the iPad for a while now, and I must
say, I am very pleased with its performance. It has met all my expectations in
terms of functionality and design. The battery life is excellent, and the
overall user experience is top-notch. I believe the iPad deserves a satisfaction
level of 3 on a scale of 1 to 4.         Regards, Jane Smith
Letter 3:
-----------------------

In [ ]:
#del generator_qwen_pipeline

### Smollm3

In [ ]:
#generator_smollm3_pipeline = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-1.7B-Instruct")

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [ ]:

#smollm3_results = generate_complaint_letters(generator_smollm3_pipeline)

#print_complaint_letters(smollm3_results)

Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Letter 1:
--------------------------------------------------
Dear Customer Service,Output the letters in the following format:
Letter 2:
--------------------------------------------------
Dear Customer Service,1. Here is your first letter. 2. Here is your second
letter. 3. Here is your third letter. 4. Here is your fourth letter. 5. Here is
your fifth letter. 6. Here is your sixth letter. 7. Here is your seventh letter.
8. Here is your eighth letter. Regards, [customer name]  Each letter's content
should be written on a new line.  Here is the code written with proper comments
to get you started:  ```python # Import necessary modules import random  #
Define a function to generate a customer letter def generate_letter(type, name):
# Define letter types and their content     complaint_types = ["unable to
connect", "poor sound quality", "unresponsive screen", "uninstalls", "unable to
charge"]     praise_types = ["excellent performance", "great screen", "stunning
performance", "very respons